In [ ]:
import sys
sys.path.insert(0, r'd:\Work\vtit_spark_transformation')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd

print("✓ Import libraries thành công")

✓ Import libraries thành công


In [2]:
spark = SparkSession.builder.appName("Test Common Utils").getOrCreate()
print("✓ Khởi tạo SparkSession thành công")

✓ Khởi tạo SparkSession thành công


## Import Common Module Functions

In [ ]:
# Import các hàm từ common module
from common.other_utils import filter_rows, sort_rows
from common.join_utils import join_dataframes, broadcast_join_dataframes, cross_join_dataframes, join_with_aliases, self_join
from common.cleaning_utils import fill_null, drop_null, coalesce_column, ifnull_column
from common.column_utils import rename_column, rename_columns, select, drop_columns, add_column, col
from common.dedup_utils import drop_duplicates
from common.window_utils import lag, lead, row_number, rank, dense_rank, ntile
from common.string_utils import lowercase, uppercase, trim, substring, concat, concat_ws, split, regexp_replace

print("✓ Nhập toàn bộ hàm từ common module thành công")

✓ Nhập toàn bộ hàm từ common module thành công


In [4]:
# Tạo dữ liệu time series
data_ts = [
    ("A", "2026-01-01", 100),
    ("A", "2026-01-02", 110),
    ("A", "2026-01-03", 120),
    ("A", "2026-01-03", 120),
    ("B", "2026-01-01", 200),
    ("B", "2026-01-02", 210),
    ("B", "2026-01-03", 215),
]
df_ts = spark.createDataFrame(data_ts, ["user_id", "date", "amount"])
df_ts.show()

+-------+----------+------+
|user_id|      date|amount|
+-------+----------+------+
|      A|2026-01-01|   100|
|      A|2026-01-02|   110|
|      A|2026-01-03|   120|
|      A|2026-01-03|   120|
|      B|2026-01-01|   200|
|      B|2026-01-02|   210|
|      B|2026-01-03|   215|
+-------+----------+------+



In [5]:
new_df = add_column(df_ts, "amount_plus_10", col(df_ts, "amount") + 10)
new_df = rename_column(new_df, "amount_plus_10", "new_amount")
new_df.show()

+-------+----------+------+----------+
|user_id|      date|amount|new_amount|
+-------+----------+------+----------+
|      A|2026-01-01|   100|       110|
|      A|2026-01-02|   110|       120|
|      A|2026-01-03|   120|       130|
|      A|2026-01-03|   120|       130|
|      B|2026-01-01|   200|       210|
|      B|2026-01-02|   210|       220|
|      B|2026-01-03|   215|       225|
+-------+----------+------+----------+



In [6]:
# Test 1: add_lag_column
print("\nTest 1: add_lag_column - Lấy giá trị ngày trước")
df_lag = lag(
    df_ts,
    source_column="amount",
    order_by=["date"],
    partition_by=["user_id"],
    offset=1,
    target_column="prev_amount"
)
df_lag.show()


Test 1: add_lag_column - Lấy giá trị ngày trước
+-------+----------+------+-----------+
|user_id|      date|amount|prev_amount|
+-------+----------+------+-----------+
|      A|2026-01-01|   100|       NULL|
|      A|2026-01-02|   110|        100|
|      A|2026-01-03|   120|        110|
|      A|2026-01-03|   120|        120|
|      B|2026-01-01|   200|       NULL|
|      B|2026-01-02|   210|        200|
|      B|2026-01-03|   215|        210|
+-------+----------+------+-----------+



In [7]:
# Test 2: add_lead_column
print("\nTest 2: add_lead_column - Lấy giá trị ngày sau")
df_lead = lead(
    df_ts,
    source_column="amount",
    order_by=[("date", False)],
    partition_by=["user_id"],
    offset=1,
    target_column="next_amount"
)
df_lead.show()


Test 2: add_lead_column - Lấy giá trị ngày sau
+-------+----------+------+-----------+
|user_id|      date|amount|next_amount|
+-------+----------+------+-----------+
|      A|2026-01-03|   120|        120|
|      A|2026-01-03|   120|        110|
|      A|2026-01-02|   110|        100|
|      A|2026-01-01|   100|       NULL|
|      B|2026-01-03|   215|        210|
|      B|2026-01-02|   210|        200|
|      B|2026-01-01|   200|       NULL|
+-------+----------+------+-----------+



In [ ]:
df2 = df_tf.low